# 01b Overlap forms confirmed from the PDFs, and gate G1

Design log.
- Replaces the nine prefilled forms under `reports/overlap/` with forms filled from the PDFs (three journal PDFs placed by hand, six arXiv PDFs downloaded by notebook 01). Every answer that points at a passage carries an anchor phrase copied from the paper.
- `src/overlap.py` gains a verification step: for each form it opens `data/surveys/<id>.pdf`, extracts the text page by page with pypdf, searches for each anchor with spacing, hyphenation and punctuation removed, writes the page where the anchor was found (correcting the stated page if needed), records the file's sha256, and marks the form confirmed only when every anchor was found. A form whose PDF is missing or whose anchor is not found stays unconfirmed and the gate stays UNRESOLVED.
- The three questions are graded with written definitions (`GRADING` in `src/overlap.py`, printed at the top of `reports/overlap_01.md`), so partial and no mean the same thing in every form.
- Decision rule unchanged: UNRESOLVED while any form is unconfirmed or invalid; NARROW if a confirmed form answers yes to any question; SUPPORTED otherwise, with the partial overlaps listed for the related-work table.
- Idempotent: rerunning rewrites the same forms and re-verifies them.
- CPU; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd())

## Stage 0b. File names in data/surveys/

In [ ]:
import corpus, paths
importlib.reload(corpus); importlib.reload(paths)
renamed = []
for p in sorted(paths.DATA_SURVEYS.glob("*.pdf.pdf")):          # hand-placed files saved with a doubled extension
    target = p.with_name(p.name[:-4])
    if target.exists():
        target.unlink()                                          # the hand-placed copy (published version) replaces the download
    p.rename(target)
    renamed.append((p.name, target.name))
for old, new in renamed:
    print(f"renamed {old} -> {new}")
rows = corpus.read_csv(paths.REPORTS / "surveys.csv")
touched = 0
for r in rows:
    pdf = paths.DATA_SURVEYS / f"{r['id']}.pdf"
    if pdf.exists():
        digest = corpus.sha256(pdf)
        if r.get("sha256") != digest:
            r["sha256"] = digest
            r["pages"] = corpus.page_count(pdf) or ""
            r["pdf_path"] = str(pdf.relative_to(paths.REPO_ROOT))
            r["pdf_status"] = "present (hand-placed)"
            touched += 1
if touched:
    corpus.write_csv(paths.REPORTS / "surveys.csv", rows, corpus.SURVEY_FIELDS)
print("surveys.csv rows refreshed:", touched)
closest = ["hong_2510.15476", "ji_2511.15203", "geng_cmc_2026", "wang_2602.10453", "sammartino_kbs_2026",
           "correia_2601.22240", "hashemi_2606.31639", "mourad_electronics_2026", "secfid_2606.30783"]
print("closest PDFs present:", sum((paths.DATA_SURVEYS / f"{i}.pdf").exists() for i in closest), "of 9;",
      "missing:", [i for i in closest if not (paths.DATA_SURVEYS / f"{i}.pdf").exists()] or "none")

## Stage 1. Module with the verification step, and its tests

In [ ]:
%%writefile src/overlap.py
"""Overlap assessment forms for the closest reviews, their verification against the PDFs, and gate G1."""
from __future__ import annotations

import hashlib
import re
import unicodedata
from pathlib import Path

import yaml

ANSWER_VALUES = {"yes", "partial", "no", "unknown"}
TEXT_FIELDS = ["scope", "corpus_cutoff", "organising_dimensions", "treatment_user_task", "treatment_document_handling",
               "treatment_comparability", "evidence_granularity", "what_this_review_adds"]
ANSWER_FIELDS = ["a_source_axis", "b_three_jobs_together", "c_equal_layout_rule"]
QUESTIONS = {
    "a_source_axis": "organises defences by how the model is shown where the instruction came from",
    "b_three_jobs_together": "reports security, the user's task and document handling together, per defence",
    "c_equal_layout_rule": "states or applies a rule that numbers are comparable only in the same prompt layout",
}
GRADING = {
    "a_source_axis": {
        "yes": "an axis whose values are forms of the source signal (chat roles, delimiters or markers, encodings, "
               "learned tokens or embeddings, position, no signal)",
        "partial": "defences are grouped by where they act or where the attack enters (a stage, level or component "
                   "axis), or signal forms appear as scattered categories but not as one axis",
        "no": "neither",
    },
    "b_three_jobs_together": {
        "yes": "security, the user's task and the handling of the document content reported together per defence",
        "partial": "two of the three reported together per defence, or their joint reporting measured as coverage",
        "no": "security alone, or utility discussed in prose or recommended but not extracted per defence",
    },
    "c_equal_layout_rule": {
        "yes": "the prompt layout (template, roles, delimiters) is named as a condition for comparing numbers",
        "partial": "a comparability condition or reporting standard is stated or applied (model, benchmark, attacker, "
                   "judge, decoding) but the layout is not among its items, or the layout is named only as a "
                   "reporting item",
        "no": "no comparability condition",
    },
}


def blank_form(record_id: str, title: str, kind: str) -> dict:
    form = {"id": record_id, "title": title, "kind": kind}
    for f in TEXT_FIELDS:
        form[f] = ""
    for f in ANSWER_FIELDS:
        form[f] = "unknown"
    form["pages"] = {"a": "", "b": "", "c": ""}
    form["anchors"] = {"a": "", "b": "", "c": ""}
    form["page_basis"] = "PDF page index, 1-based, of the file in data/surveys/"
    form["read_basis"] = ""
    form["confirmed_from_pdf"] = False
    form["pdf_sha256"] = ""
    form["verification"] = ""
    return form


def write_form(path: Path, form: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    clean = {k: v for k, v in form.items() if not k.startswith("_")}
    path.write_text(yaml.safe_dump(clean, sort_keys=False, allow_unicode=True, width=110))


def write_form_if_missing(path: Path, form: dict) -> bool:
    if Path(path).exists():
        return False
    write_form(path, form)
    return True


def load_forms(folder: Path) -> list[dict]:
    forms = []
    for path in sorted(Path(folder).glob("*.yaml")):
        form = yaml.safe_load(path.read_text()) or {}
        form["_path"] = str(path)
        forms.append(form)
    return forms


def validate(form: dict) -> list[str]:
    problems = []
    for f in ["id", "title", "kind", "read_basis"]:
        if not form.get(f):
            problems.append(f"{f} missing")
    for f in TEXT_FIELDS:
        if not str(form.get(f, "")).strip():
            problems.append(f"{f} empty")
    for f in ANSWER_FIELDS:
        if form.get(f) not in ANSWER_VALUES:
            problems.append(f"{f} must be one of {sorted(ANSWER_VALUES)}")
    if form.get("confirmed_from_pdf"):
        if not form.get("pdf_sha256"):
            problems.append("confirmed_from_pdf is true but pdf_sha256 is empty")
        for key, f in zip("abc", ANSWER_FIELDS):
            if form.get(f) in {"yes", "partial"} and not str((form.get("pages") or {}).get(key, "")).strip():
                problems.append(f"{f} is {form.get(f)} but pages.{key} is empty")
        if any(form.get(f) == "unknown" for f in ANSWER_FIELDS):
            problems.append("confirmed_from_pdf is true but an answer is still unknown")
    return problems


# ---- verification against the PDF on disk -------------------------------------------------------------------

def norm_text(s: str) -> str:
    """Lower-case alphanumerics only, so spacing, hyphenation, ligatures and punctuation cannot break a match."""
    s = unicodedata.normalize("NFKC", s or "").replace("­", "")
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def pdf_page_texts(path: Path) -> list[str]:
    from pypdf import PdfReader
    return [norm_text(page.extract_text() or "") for page in PdfReader(str(path)).pages]


def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def verify_form(form: dict, pdf_path: Path, page_texts: list[str] | None = None) -> dict:
    """Locate each anchor phrase in the PDF, correct the page to where it was found, and set confirmed_from_pdf.

    An anchor is a short phrase copied from the paper. A question with an empty anchor is not checked; one whose
    anchor is not found leaves the form unconfirmed. Page numbers are PDF page indices (1-based).
    """
    pdf_path = Path(pdf_path)
    notes = []
    if not pdf_path.exists():
        form["confirmed_from_pdf"] = False
        form["verification"] = "PDF missing"
        return form
    texts = page_texts if page_texts is not None else pdf_page_texts(pdf_path)
    form.setdefault("pages", {}); form.setdefault("anchors", {})
    all_found = True
    for key in "abc":
        anchor = (form["anchors"] or {}).get(key, "")
        if not anchor:
            continue
        hits = [i + 1 for i, t in enumerate(texts) if norm_text(anchor) in t]
        stated = str(form["pages"].get(key, "")).strip()
        if not hits:
            all_found = False
            notes.append(f"{key}: anchor not found")
            continue
        if stated and stated.isdigit() and int(stated) in hits:
            notes.append(f"{key}: p.{stated} ok")
        else:
            notes.append(f"{key}: p.{stated or '?'} -> p.{hits[0]}" + (f" (also {hits[1:]})" if len(hits) > 1 else ""))
            form["pages"][key] = str(hits[0])
    form["pdf_sha256"] = sha256_of(pdf_path)
    form["pdf_pages"] = len(texts)
    form["confirmed_from_pdf"] = all_found
    form["verification"] = "; ".join(notes) if notes else "no anchors to check"
    return form


# ---- gate G1 ---------------------------------------------------------------------------------------------------

def decide(forms: list[dict]) -> tuple[str, list[str]]:
    """UNRESOLVED if any form is unconfirmed or invalid; NARROW if any confirmed form answers yes; else SUPPORTED."""
    reasons = []
    unconfirmed = [f["id"] for f in forms if not f.get("confirmed_from_pdf")]
    invalid = {f["id"]: validate(f) for f in forms}
    invalid = {k: v for k, v in invalid.items() if v}
    if invalid:
        reasons += [f"{k}: {'; '.join(v)}" for k, v in invalid.items()]
    if unconfirmed:
        reasons.append("not yet confirmed from the PDF: " + ", ".join(unconfirmed))
    if invalid or unconfirmed:
        return "UNRESOLVED", reasons
    narrow = []
    for f in forms:
        for field in ANSWER_FIELDS:
            if f.get(field) == "yes":
                narrow.append(f"{f['id']} answers yes to {field} ({QUESTIONS[field]}); that claim must be narrowed or dropped")
    if narrow:
        return "NARROW", narrow
    partials = [f"{f['id']} ({', '.join(k for k in ANSWER_FIELDS if f.get(k) == 'partial')})"
                for f in forms if any(f.get(k) == "partial" for k in ANSWER_FIELDS)]
    reasons = ["no confirmed form answers yes to any of the three questions"]
    if partials:
        reasons.append("partial overlaps to be named in the related-work table: " + "; ".join(partials))
    return "SUPPORTED", reasons


def render_markdown(forms: list[dict], outcome: str, reasons: list[str], when: str) -> str:
    lines = [f"# Overlap assessment (gate G1), {when}", "", f"Outcome: **{outcome}**", ""]
    lines += [f"- {r}" for r in reasons] + [""]
    lines += ["## How the three questions were graded", ""]
    for field in ANSWER_FIELDS:
        lines.append(f"**{field}**: {QUESTIONS[field]}.")
        for value in ("yes", "partial", "no"):
            lines.append(f"- {value}: {GRADING[field][value]}")
        lines.append("")
    lines += ["Pages are 1-based page indices of the PDF in `data/surveys/`; each page was located by searching the PDF "
              "text for the anchor phrase recorded in the form (`verification` field).", ""]
    lines += ["| id | kind | a: source axis | b: three jobs | c: layout rule | pages a / b / c | confirmed | read basis |",
              "|---|---|---|---|---|---|---|---|"]
    for f in forms:
        p = f.get("pages") or {}
        lines.append(f"| {f.get('id')} | {f.get('kind')} | {f.get('a_source_axis')} | {f.get('b_three_jobs_together')} | "
                     f"{f.get('c_equal_layout_rule')} | {p.get('a', '')} / {p.get('b', '')} / {p.get('c', '')} | "
                     f"{'yes' if f.get('confirmed_from_pdf') else 'no'} | {f.get('read_basis')} |")
    lines.append("")
    for f in forms:
        lines += [f"## {f.get('id')}: {f.get('title')}", ""]
        for field in TEXT_FIELDS:
            lines.append(f"- **{field.replace('_', ' ')}**: {f.get(field, '')}")
        anchors = f.get("anchors") or {}
        pages = f.get("pages") or {}
        for key, field in zip("abc", ANSWER_FIELDS):
            if anchors.get(key):
                lines.append(f"- **{field}** = {f.get(field)}; p.{pages.get(key, '')}: \"{anchors.get(key)}\"")
            else:
                lines.append(f"- **{field}** = {f.get(field)}; no passage to point to")
        lines.append(f"- **verification**: {f.get('verification', '')}; sha256 {str(f.get('pdf_sha256', ''))[:12]}")
        lines.append("")
    return "\n".join(lines)

In [ ]:
%%writefile tests/test_overlap.py

"""The gate G1 decision rule and the anchor verification."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
import overlap  # noqa: E402


def confirmed(record_id, a="no", b="no", c="no"):
    form = overlap.blank_form(record_id, "t", "survey")
    for f in overlap.TEXT_FIELDS:
        form[f] = "x"
    form.update(a_source_axis=a, b_three_jobs_together=b, c_equal_layout_rule=c, read_basis="full PDF",
                confirmed_from_pdf=True, pdf_sha256="abc")
    form["pages"] = {"a": "3", "b": "4", "c": "5"}
    return form


def test_unresolved_when_a_form_is_unconfirmed():
    forms = [confirmed("x"), overlap.blank_form("y", "t", "survey")]
    outcome, reasons = overlap.decide(forms)
    assert outcome == "UNRESOLVED"
    assert any("not yet confirmed" in r for r in reasons)


def test_narrow_when_a_confirmed_form_answers_yes():
    outcome, reasons = overlap.decide([confirmed("x"), confirmed("y", b="yes")])
    assert outcome == "NARROW"
    assert "b_three_jobs_together" in reasons[0]


def test_supported_lists_partial_overlaps():
    outcome, reasons = overlap.decide([confirmed("x", a="partial"), confirmed("y", c="partial")])
    assert outcome == "SUPPORTED"
    assert any("x (a_source_axis)" in r and "y (c_equal_layout_rule)" in r for r in reasons)


def test_confirmed_form_needs_pages_for_partial_answers():
    form = confirmed("x", a="partial")
    form["pages"]["a"] = ""
    assert any("pages.a" in p for p in overlap.validate(form))


def test_norm_text_ignores_spacing_hyphenation_and_ligatures():
    assert overlap.norm_text("Defense-to-threat   cover-\nage map") == "defensetothreatcoverage map".replace(" ", "")
    assert overlap.norm_text("\ufb01delity") == "fidelity"
    assert overlap.norm_text("t h er e s e a r c h") == overlap.norm_text("the research")


def test_verify_form_corrects_page_and_confirms(tmp_path):
    pdf = tmp_path / "x.pdf"
    pdf.write_bytes(b"%PDF-1.4 fake")
    pages = [overlap.norm_text(t) for t in ["intro", "we define fidelity as 1 - ignored rate", "structural separation"]]
    form = confirmed("x", a="partial", b="partial")
    form["anchors"] = {"a": "structural separation", "b": "Fidelity as 1 - Ignored Rate", "c": ""}
    form["pages"] = {"a": "2", "b": "2", "c": ""}
    out = overlap.verify_form(form, pdf, page_texts=pages)
    assert out["confirmed_from_pdf"] is True
    assert out["pages"] == {"a": "3", "b": "2", "c": ""}
    assert out["pdf_sha256"] and out["pdf_pages"] == 3
    assert "a: p.2 -> p.3" in out["verification"] and "b: p.2 ok" in out["verification"]


def test_verify_form_leaves_unconfirmed_when_anchor_missing_or_pdf_missing(tmp_path):
    pdf = tmp_path / "x.pdf"
    pdf.write_bytes(b"%PDF-1.4 fake")
    form = confirmed("x")
    form["anchors"] = {"a": "phrase that is not there", "b": "", "c": ""}
    out = overlap.verify_form(form, pdf, page_texts=["nothing"])
    assert out["confirmed_from_pdf"] is False and "a: anchor not found" in out["verification"]
    gone = overlap.verify_form(confirmed("y"), tmp_path / "missing.pdf")
    assert gone["confirmed_from_pdf"] is False and gone["verification"] == "PDF missing"

## Stage 2. The nine forms, filled from the PDFs

In [ ]:
import yaml
import corpus, overlap, paths
importlib.reload(overlap); importlib.reload(paths)

FORMS = {'sammartino_kbs_2026': {'scope': 'PRISMA-informed review of attacks and defences in LLM-driven agentic and '
                                  'multi-agent systems; 102-paper corpus (44 attack papers, 31 defence papers); '
                                  'window 1 January 2023 to 31 January 2026 (p. 4, p. 7).',
                         'corpus_cutoff': '31 January 2026 (search closed; p. 7, p. 21)',
                         'organising_dimensions': 'Threats by entry point, propagation path and target asset '
                                                  '(Section 4, pp. 7-9). Defences in five families ordered by the '
                                                  'taxonomy dimension they act on: detection and filtering; '
                                                  'structural separation of instructions and data (StruQ, SecAlign, '
                                                  'Jatmo, spotlighting in one family); execution isolation and least '
                                                  'privilege; information-flow control; alignment-side and '
                                                  'multi-agent hardening (Section 8, pp. 13-17). A defence-to-threat '
                                                  'coverage map graded by evidence type: containment argument, '
                                                  'empirical reduction, principled extension, not addressed (Section '
                                                  '8.1 and Table 4, pp. 13 and 18).',
                         'treatment_user_task': "Security and the user's task together is the paper's main "
                                                'methodological point: ASR reported alone is called uninterpretable '
                                                'and the joint (ASR, utility) point is called the only result '
                                                '(Section 9.2, p. 17). Pairing is measured as coverage: 25 of 44 '
                                                'attack papers and 12 of 31 defence papers report no benign-utility '
                                                'measurement (quality item Q3, p. 5; Table 3, pp. 15-16); Table 5 '
                                                "carries a utility-cost column per family (p. 19). The user's task "
                                                'under attack is not separated from clean utility.',
                         'treatment_document_handling': 'Not measured. Whether a defended model still reads the '
                                                        'untrusted content is not among the quality items, the '
                                                        'tables or the reporting standard.',
                         'treatment_comparability': 'Section 9.3 names threat-model incommensurability (attacker '
                                                    'capability, defender scope). The reporting standard of Section '
                                                    '11.7 (p. 21) asks for attacker capability, defender scope, '
                                                    'paired (ASR, utility), adaptivity, judge, cost, taxonomic '
                                                    'coordinates, a reproducibility block (checkpoint, decoding, '
                                                    'seeds, runs) and residual risk. The prompt layout or template '
                                                    'is not an item.',
                         'evidence_granularity': 'Per attack paper: ASR as reported, paired utility (value, '
                                                 'per-configuration or not reported), models, environment (Table 3). '
                                                 'Per defence family, not per defence: adaptive resistance, utility '
                                                 'cost, deployment cost, residual risk, evidence grade (Table 5). No '
                                                 'per-defence numbers are extracted.',
                         'what_this_review_adds': 'Model-side defences at the level of individual defences rather '
                                                  'than families; the signal axis with its no-signal value; the '
                                                  "user's task measured under attack and the document-handling "
                                                  'requirement, which this paper does not have; the prompt layout as '
                                                  'a comparison condition added to a reporting list that already has '
                                                  'checkpoint, decoding, judge and attacker; and an evidence grade '
                                                  'for each pairwise comparison rather than for each family-threat '
                                                  "cell. This paper's utility-trap argument and Q3 coverage count "
                                                  'are the closest prior statement of the pairing requirement and '
                                                  'must be cited as such.',
                         'a_source_axis': 'partial',
                         'b_three_jobs_together': 'partial',
                         'c_equal_layout_rule': 'partial',
                         'pages': {'a': '14', 'b': '17', 'c': '21'},
                         'anchors': {'a': 'structural separation of instructions and data',
                                     'b': 'Reported alone, ASR is uninterpretable',
                                     'c': 'Defender scope. What the defense may modify'},
                         'read_basis': 'full PDF, Knowledge-Based Systems 352 (2026) 117001, 25 pages, read 20 Nov '
                                       '2026'},
 'mourad_electronics_2026': {'scope': 'Systematic review of defence mechanisms against prompt-based attacks '
                                      '(jailbreak, direct and indirect injection) in LLMs and LLM agents; 19 studies '
                                      'from ACM and IEEE, 2023-2025 (Figure 2, p. 6; Table 2, p. 8).',
                             'corpus_cutoff': '2025 (inclusion criterion I3, p. 8)',
                             'organising_dimensions': 'System architecture (single-LLM, multi-agent); defence '
                                                      'methodology (ML-based, rule-based, hybrid); pipeline stage at '
                                                      'which the defence applies (prompt-, model-, pipeline-level) '
                                                      '(Section 1.1, p. 3; Section 6, p. 12; RQ6, p. 11). Entry '
                                                      'sources of malicious text are an attack question (RQ2, p. '
                                                      '11). Fixed delimiters and guardrail text appear as one '
                                                      'rule-based technique (RQ7, p. 11).',
                             'treatment_user_task': 'Listed among reported metrics in prose: latency, throughput and '
                                                    'task accuracy degradation (RQ9, p. 12); a trade-off paragraph '
                                                    '(Section 8.2, p. 23); a recommendation to report ASR alongside '
                                                    'a utility metric (Section 8.3, p. 24). Not extracted per study.',
                             'treatment_document_handling': 'Not treated.',
                             'treatment_comparability': 'States that reported attack success rates are not directly '
                                                        'comparable across studies because of differing datasets, '
                                                        'threat models and reporting conventions, and declines to '
                                                        'build a unified comparison table (p. 14). Layout or '
                                                        'template is not named.',
                             'evidence_granularity': 'Per study: attack type, defence, dataset, metrics, '
                                                     'architecture (Tables 6 and 7); quality scores (Table 5). No '
                                                     'numbers compared across studies.',
                             'what_this_review_adds': 'A corpus of model-side defences against indirect injection '
                                                      "(this paper's 19 studies are mostly jailbreak detectors); the "
                                                      'two axes; per-defence measurement coverage of the three '
                                                      'requirements; the layout condition and graded comparisons in '
                                                      'place of a refusal to compare.',
                             'a_source_axis': 'partial',
                             'b_three_jobs_together': 'no',
                             'c_equal_layout_rule': 'partial',
                             'pages': {'a': '11', 'b': '12', 'c': '14'},
                             'anchors': {'a': 'fixed delimiters or guardrail text that separate system instructions '
                                              'from user input',
                                         'b': 'Utility-related metrics such as latency, throughput, and task '
                                              'accuracy degradation',
                                         'c': 'not directly comparable across studies because of differing datasets, '
                                              'threat models'},
                             'read_basis': 'full PDF, Electronics 15(16):3700, 27 pages, read 20 Nov 2026'},
 'geng_cmc_2026': {'scope': 'Kitchenham-style systematic review of prompt injection attacks, root causes and '
                            'defences; 128 studies 2022-2025; 37 defence studies tabulated (abstract, p. 1; Table 6, '
                            'pp. 28-30).',
                   'corpus_cutoff': '2025 (p. 1, p. 28)',
                   'organising_dimensions': 'Attacks by vector, objective and implementation (Section 3). Root '
                                            'causes at philosophical, technical and training levels (Section 4). '
                                            'Defences in three categories: input preprocessing and filtering, system '
                                            'architecture, model-level (Section 5, p. 28). Spotlighting, '
                                            'DefensiveToken, StruQ, SecAlign are placed in these categories by where '
                                            'they act, not by how the source is shown.',
                   'treatment_user_task': 'Not extracted. Table 7 (p. 33) lists baseline ASR and ASR with defence '
                                          'only, for seven defences on different benchmarks and models. A unified '
                                          'evaluation framework is recommended with three dimensions: security '
                                          '(ASR), efficiency, usability (false positive rate, task completion '
                                          'accuracy) (Section 6.3, pp. 41-42).',
                   'treatment_document_handling': 'Not treated.',
                   'treatment_comparability': 'States that studies use varying datasets, metrics and settings, '
                                              'giving incomparable results (p. 41), yet Table 7 (p. 33) places ASR '
                                              'figures from different benchmarks and models side by side. Layout is '
                                              'not named.',
                   'evidence_granularity': 'Per defence: year, category, authors, one-line strategy (Table 6); ASR '
                                           'before and after for seven defences (Table 7); datasets and metrics '
                                           '(Table 9).',
                   'what_this_review_adds': 'The signal axis; measurement coverage of the three requirements per '
                                            'defence; comparisons admitted only with a grade, which would mark Table '
                                            "7's juxtaposition as not comparable; the layout condition.",
                   'a_source_axis': 'partial',
                   'b_three_jobs_together': 'no',
                   'c_equal_layout_rule': 'partial',
                   'pages': {'a': '28', 'b': '33', 'c': '41'},
                   'anchors': {'a': 'Input Preprocessing and Filtering, System Architecture Defenses, and '
                                    'Model-Level Defenses',
                               'b': 'Quantitative comparison of defense mechanism effectiveness',
                               'c': 'leading to inconsistent and sometimes incomparable results'},
                   'read_basis': 'full PDF, Computers, Materials & Continua 87(1), 52 pages, read 20 Nov 2026'},
 'hashemi_2606.31639': {'scope': 'Lifecycle and application-stack survey of LLM vulnerabilities; systematisation '
                                 'from seed papers and citation chasing, cross-checked against OWASP, MITRE ATLAS '
                                 'and NIST; no paper count or search dates (Section 2.2, p. 2).',
                        'corpus_cutoff': 'not stated; arXiv v1 30 June 2026',
                        'organising_dimensions': 'Attacks by lifecycle stage and application stack (Section 3). '
                                                 'Defences in a defence-in-depth architecture: deterministic '
                                                 'controls; model-level and prompt-level controls; retrieval and '
                                                 'memory controls; tool and agent controls; monitoring, red teaming '
                                                 'and governance (Section 6, pp. 9-10). Delimiters, role labels and '
                                                 'instruction hierarchy are listed as examples of prompt-level '
                                                 'controls, not as an axis.',
                        'treatment_user_task': 'Named as metrics to report (clean-task accuracy, user-task utility; '
                                               'Section 7, p. 10); no numbers for any defence.',
                        'treatment_document_handling': 'Named as a design goal: a model should read untrusted '
                                                       'content for information without letting it change goals '
                                                       '(Section 9, p. 13); not measured.',
                        'treatment_comparability': 'A reporting checklist (Table 5, p. 11) includes context '
                                                   'construction: system prompt, retrieved documents, memory policy, '
                                                   'delimiters, tool schemas, with the rationale that it is critical '
                                                   'for prompt-injection and RAG studies. It is a reporting item; no '
                                                   'rule says numbers are comparable only under the same layout.',
                        'evidence_granularity': 'Narrative with reference tables; no extracted per-defence evidence.',
                        'what_this_review_adds': 'Filled records rather than a checklist: the signal axis, '
                                                 'per-defence coverage of the three requirements, layout as a '
                                                 'comparison condition with graded comparisons, and the defence '
                                                 "card. This paper's Table 5 is the nearest prior naming of "
                                                 'delimiters and templates as something to report.',
                        'a_source_axis': 'partial',
                        'b_three_jobs_together': 'no',
                        'c_equal_layout_rule': 'partial',
                        'pages': {'a': '10', 'b': '10', 'c': '11'},
                        'anchors': {'a': 'system prompts, instruction hierarchy, delimiters, reminders about '
                                         'untrusted content',
                                    'b': 'clean-task accuracy',
                                    'c': 'critical for prompt-injection and RAG studies'},
                        'read_basis': 'full PDF text, arXiv 2606.31639 (latest version on 20 Nov 2026), 19 pages, '
                                      'read 20 Nov 2026'},
 'correia_2601.22240': {'scope': 'Systematic literature review of LLM defences against prompt injection and '
                                 'jailbreaking, extending the NIST taxonomy; 88 primary studies from Scopus; states '
                                 'it was submitted to Computer Science Review (p. 1, p. 6).',
                        'corpus_cutoff': 'August 2025 (p. 6)',
                        'organising_dimensions': 'Training-time interventions (post-training, pre-training); '
                                                 'deployment-time interventions (prompt instruction and formatting, '
                                                 'input modification, aggregation, prompt-stealing prevention, '
                                                 'model-level mitigations, detection and termination, monitoring, '
                                                 'usage restrictions); indirect mitigations; mitigations against '
                                                 'indirect prompt injection; evaluation-time interventions (Figure '
                                                 '1, p. 2; Sections 5.1-5.5). Delimiters sit under prompt '
                                                 'instruction and formatting (p. 9); instruction hierarchy in '
                                                 'embeddings and spotlighting under indirect-injection mitigations '
                                                 '(p. 14).',
                        'treatment_user_task': 'Not extracted. The catalogue columns are method, description, '
                                               'model-agnostic, attack mitigation (mostly ASR, occasionally a '
                                               'false-positive value), attack, source code (Tables A.5-A.11). States '
                                               'that many works do not report trade-offs such as overall performance '
                                               'degradation (p. 15).',
                        'treatment_document_handling': 'Not treated beyond row descriptions.',
                        'treatment_comparability': 'RQ3 asks whether results are comparable (p. 2). The answer names '
                                                   'dataset splits, model variants and parameters such as '
                                                   'temperature as what prevents comparison (pp. 15-16). Layout or '
                                                   'template is not named.',
                        'evidence_granularity': 'Per study catalogue rows (Tables A.5-A.11) with ASR where reported; '
                                                'no cross-study numbers.',
                        'what_this_review_adds': "The signal axis separated from the training axis (this paper's top "
                                                 'split is training-time versus deployment-time, a coarse form of '
                                                 'the second axis only); the three requirements per defence; layout '
                                                 'added to the comparability factors; graded comparisons.',
                        'a_source_axis': 'partial',
                        'b_three_jobs_together': 'no',
                        'c_equal_layout_rule': 'partial',
                        'pages': {'a': '9', 'b': '15', 'c': '16'},
                        'anchors': {'a': 'Enclosing queries within random characters or XML',
                                    'b': 'Many works do not report trade-offs such as overall performance '
                                         'degradation',
                                    'c': 'dataset splits, model variants, and parameters like temperature'},
                        'read_basis': 'full PDF text, arXiv 2601.22240 (latest version on 20 Nov 2026), 27 pages, '
                                      'read 20 Nov 2026'},
 'wang_2602.10453': {'scope': 'SoK of prompt injection threats in LLM agents; 78 papers (37 attack, 41 defence); '
                              'search ended 20 October 2025 (p. 1; pp. 4 and 6).',
                     'corpus_cutoff': '20 October 2025 (p. 1)',
                     'organising_dimensions': 'Defences by level: text-level (detection filters; prompt enhancement '
                                              'by separation, negative prompts, rewriting), model-level (alignment '
                                              'by fine-tuning; internal-representation probes), execution-level '
                                              '(task alignment, access control, isolation) (Figure 2, p. 5; Table 2, '
                                              "p. 6; Sections 4.1-4.3, pp. 7-9). The 'separate' family uses "
                                              'delimiters and structure, and alignment trains on specific '
                                              'interaction formats, but there is no axis over signal forms.',
                     'treatment_user_task': 'Eight defences run on AgentPI with GPT-4o-mini: ASR and utility under '
                                            'no attack (Figure 6, p. 12), time and tokens (Table 5, p. 13). Utility '
                                            'under attack is not reported.',
                     'treatment_document_handling': 'Not measured. The paper states that many defences appear '
                                                    'effective by suppressing contextual inputs (abstract, p. 1) and '
                                                    'by coarse-grained refusal (p. 11), which is the symptom this '
                                                    "review's third requirement measures.",
                     'treatment_comparability': 'All defences run on one model and one benchmark, an applied '
                                                'condition; a change of prompt structure is counted as an '
                                                'intrusiveness cost (p. 10), not as a condition for comparing '
                                                'numbers.',
                     'evidence_granularity': 'Per defence on one model: ASR, no-attack utility, time, tokens (Figure '
                                             '6, Table 5); the model-level defences are not run.',
                     'what_this_review_adds': "The signal axis with the no-signal value; the user's task under "
                                              'attack and the document-handling requirement measured, which turns '
                                              "this paper's suppression remark into a number; layout as a comparison "
                                              'condition; graded comparisons across published numbers including '
                                              'model-side defences.',
                     'a_source_axis': 'partial',
                     'b_three_jobs_together': 'no',
                     'c_equal_layout_rule': 'partial',
                     'pages': {'a': '7', 'b': '12', 'c': '10'},
                     'anchors': {'a': 'uses structural transformations and delimiters',
                                 'b': 'utility under no attack cases',
                                 'c': 'require a change of the prompt structure'},
                     'read_basis': 'full PDF text, arXiv 2602.10453 (latest version on 20 Nov 2026), 20 pages, read '
                                   '20 Nov 2026'},
 'ji_2511.15203': {'scope': 'Taxonomy, evaluation and exploitation of defence frameworks for indirect prompt '
                            'injection in LLM agents; 26 frameworks in the taxonomy (Table 1), nine evaluated '
                            '(Section 4).',
                   'corpus_cutoff': 'arXiv v1 19 November 2025; no search window stated',
                   'organising_dimensions': 'Five dimensions: technical paradigm (detection, prompt engineering, '
                                            'fine-tuning, system design, runtime checking, policy enforcing), '
                                            'intervention stage, model access, explainability, automation level '
                                            '(Section 3, Table 1). Delimiters, hash tags and defensive tokens are '
                                            'described inside the prompt-engineering paradigm, not as a dimension.',
                   'treatment_user_task': 'Utility measured as the proportion of user-specified tasks completed in '
                                          'AgentDojo without attack (Section 4.1, Table 3); false positives '
                                          '(Appendix B); time and tokens (Table 3). Utility under attack is not '
                                          'measured.',
                   'treatment_document_handling': 'Not measured.',
                   'treatment_comparability': 'Configurations standardised across benchmarks and frameworks: GPT-4o '
                                              'backend, temperature 0, OpenAI function-calling template (Section 4); '
                                              'fine-tuned frameworks run on Llama-3.1-8B instead, and frameworks '
                                              'needing manual configuration are run on their automated parts only. '
                                              'An applied equal-conditions rule for the backbone and tool template; '
                                              "each framework's own prompt layout is not held equal and is not "
                                              'described.',
                   'evidence_granularity': 'Per framework: ASR on AgentDojo, ASB and InjecAgent under benchmark and '
                                           'adaptive attacks (Table 2); utility, false positives, cost (Table 3, '
                                           'Table 8).',
                   'what_this_review_adds': 'Model-side defences as the subject (this paper runs one trained '
                                            "defence, on a different model from the rest); the two axes; the user's "
                                            'task under attack and document handling; the layout condition made '
                                            "explicit; graded comparisons rather than one platform's run.",
                   'a_source_axis': 'partial',
                   'b_three_jobs_together': 'no',
                   'c_equal_layout_rule': 'partial',
                   'pages': {'a': '', 'b': '', 'c': ''},
                   'anchors': {'a': 'unpredictable data delimiters',
                               'b': 'the proportion of user-specified tasks completed in the AgentDojo benchmark',
                               'c': 'we standardize the configurations across all benchmarks and defense frameworks'},
                   'read_basis': 'full PDF text, arXiv 2511.15203 (latest version on 20 Nov 2026); page numbers '
                                 'located by anchor search, read 20 Nov 2026'},
 'hong_2510.15476': {'scope': 'SoK of LLM prompt security centred on jailbreaks: 272 papers, collection last updated '
                              'May 2026; three taxonomies and a unified evaluation platform (Sections 1, 2.1, 3). '
                              'States that it focuses on textual jailbreak attacks and defences (Section 1); the '
                              'platform supports single-turn interactions only (Section 3.2).',
                     'corpus_cutoff': 'May 2026 (Section 2.1)',
                     'organising_dimensions': 'Defence taxonomy by goal (detection, mitigation), then intervention '
                                              'point (input, output, inner state; input processing, model training, '
                                              'model-side intervention, output processing), then operation (Section '
                                              '2.5, Figure 2). Provenance marking and delimiting are techniques '
                                              'under prompt modification; safety prompts with optimised embeddings '
                                              'sit beside them. No axis over signal forms.',
                     'treatment_user_task': 'Per defence under one protocol: ASR, defence gain, sample-wise '
                                            'transitions (Table 2, Section 4.1); benign utility on MMLU, TruthfulQA, '
                                            'FollowBench, IFEval and refusal rate, with over-refusal relative to no '
                                            'defence (Section 4.4). This is general capability without attack, not '
                                            "the user's task under attack.",
                     'treatment_document_handling': 'Not measured; the evaluated attacks are direct harmful queries, '
                                                    'not instructions carried in documents or tool outputs.',
                     'treatment_comparability': 'States that two results are directly comparable only when component '
                                                'identities and relevant settings match, over the tuple model, '
                                                'attack, defence, dataset, judge plus decoding, assistant models, '
                                                'budgets, thresholds, judge rubrics and parsing rules (Section 3.1). '
                                                'The strongest comparability rule among the surveys; the prompt '
                                                'layout is not a listed element.',
                     'evidence_granularity': '11 models, 20 attacks, 9 defences, 3 judges run under one protocol '
                                             'with audit logs.',
                     'what_this_review_adds': "Indirect injection as the subject; the signal axis; the user's task "
                                              'under attack and document handling; the layout condition added to a '
                                              'comparability rule that this paper already states for model, attack, '
                                              'dataset, judge and decoding; grades for published comparisons instead '
                                              'of a re-run.',
                     'a_source_axis': 'partial',
                     'b_three_jobs_together': 'partial',
                     'c_equal_layout_rule': 'partial',
                     'pages': {'a': '', 'b': '', 'c': ''},
                     'anchors': {'a': 'provenance marking, delimiting',
                                 'b': 'defense-induced over-refusal relative to no_defense',
                                 'c': 'Two results are directly comparable only when both component identities and '
                                      'relevant settings match'},
                     'read_basis': 'full PDF text, arXiv 2510.15476 (latest version on 20 Nov 2026); page numbers '
                                   'located by anchor search, read 20 Nov 2026'},
 'secfid_2606.30783': {'scope': 'Benchmark and evaluation paper, not a survey: 1,168 instances (plus a 252-scenario '
                                'agentic extension), four defences in eight variants over 15 base-model settings, 48 '
                                'configurations (Sections 1, 4); ICML 2026, PMLR 306.',
                       'corpus_cutoff': 'not applicable',
                       'organising_dimensions': 'Related work groups defences by intervention point: input-centric '
                                                'filtering, prompting-level delimiters, model-centric fine-tuning or '
                                                "architecture (Section 6). The paper's own structure is the outcome "
                                                'coding: executed, processed, ignored, other (Section 3.4).',
                       'treatment_user_task': 'Not measured as a separate quantity; the task in each instance is the '
                                              'document task itself.',
                       'treatment_document_handling': 'Measured and defined: security is one minus the executed '
                                                      'rate; fidelity is one minus the ignored rate; safe processing '
                                                      'is processed and not executed (Section 3.5). On Llama 3.1 8B '
                                                      '(Table 1): executed and ignored rates of 49.1/7.7 undefended, '
                                                      '5.7/24.7 ASIDE, 14.0/20.3 ISE, 0.7/26.1 SecAlign, 1.9/53.7 '
                                                      'DefensiveTokens. Section 4.2 text gives 1.2 and 3.0 for the '
                                                      'last two where Table 1 gives 0.7 and 1.9; cite Table 1 and '
                                                      'Table 22 (confidence intervals), not the prose.',
                       'treatment_comparability': 'Each defence is run with the inference procedure its authors '
                                                  'specify (Appendix B.1); ASIDE and ISE from the official ASIDE '
                                                  'repository. No comparability rule; layouts therefore differ '
                                                  'across the compared defences.',
                       'evidence_granularity': 'Per defence and base model: executed, ignored, processed rates with '
                                               'confidence intervals.',
                       'what_this_review_adds': 'SecFid is the source of the document-handling definitions and of '
                                                'published security-fidelity pairs; this review places those pairs '
                                                "on the two-axis map with their layouts recorded, adds the user's "
                                                'task, and grades every comparison for layout equality, which '
                                                "SecFid's own protocol does not hold.",
                       'a_source_axis': 'no',
                       'b_three_jobs_together': 'partial',
                       'c_equal_layout_rule': 'no',
                       'pages': {'a': '', 'b': '', 'c': ''},
                       'anchors': {'a': 'Input-centric methods attempt to filter malicious content before it reaches '
                                        'the model',
                                   'b': 'We define Fidelity as 1 - Ignored Rate',
                                   'c': 'we use the inference procedure specified by each defense'},
                       'read_basis': 'full PDF text, arXiv 2606.30783 (latest version on 20 Nov 2026); page numbers '
                                     'located by anchor search, read 20 Nov 2026'}}

by_id = {r["id"]: r for r in corpus.read_csv(paths.REPORTS / "surveys.csv")}
written = []
for record_id, values in FORMS.items():
    rec = by_id.get(record_id, {})
    form = overlap.blank_form(record_id, rec.get("title", ""), rec.get("kind", ""))
    form.update(values)
    form["confirmed_from_pdf"] = False
    form["pdf_sha256"] = ""
    overlap.write_form(paths.REPORTS / "overlap" / f"{record_id}.yaml", form)
    written.append(record_id)
print("forms written:", len(written))
missing_title = [i for i in written if not by_id.get(i, {}).get("title")]
print("ids not found in reports/surveys.csv (run notebook 01 first):", missing_title or "none")

## Stage 3. Verification of every anchor against the PDF on disk

In [ ]:
forms = overlap.load_forms(paths.REPORTS / "overlap")
rows = []
for form in forms:
    pdf = paths.DATA_SURVEYS / f"{form['id']}.pdf"
    overlap.verify_form(form, pdf)
    overlap.write_form(Path(form["_path"]), form)
    rows.append((form["id"], "yes" if form["confirmed_from_pdf"] else "NO", form.get("pdf_pages", ""), form["verification"]))
width = max(len(r[0]) for r in rows)
for r in rows:
    print(f"{r[0]:<{width}}  confirmed={r[1]:<3} pages={str(r[2]):<4} {r[3]}")
print()
print("confirmed:", sum(1 for r in rows if r[1] == "yes"), "of", len(rows))

## Stage 4. Gate G1

In [ ]:
import datetime as dt
forms = overlap.load_forms(paths.REPORTS / "overlap")
outcome, reasons = overlap.decide(forms)
when = dt.date.today().isoformat()
(paths.REPORTS / "overlap_01.md").write_text(overlap.render_markdown(forms, outcome, reasons, when))
lines = [f"decision_01: {outcome}", f"date: {when}", f"forms: {len(forms)}"] + [f"- {r}" for r in reasons]
(paths.REPORTS / "decision_01.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
print()
print("| id | a | b | c | pages a/b/c |")
for f in forms:
    p = f.get("pages") or {}
    print(f"| {f['id']} | {f['a_source_axis']} | {f['b_three_jobs_together']} | {f['c_equal_layout_rule']} | {p.get('a','')}/{p.get('b','')}/{p.get('c','')} |")
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print()
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 5. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "01b overlap confirmation: nine forms read from the PDFs, anchor verification, gate G1 decision"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)